In [33]:
from collections import OrderedDict
from copy import deepcopy
from functools import reduce

import matplotlib.pyplot as plt
import numpy as np
import panel as pn
import plotly.colors as pc
import plotly.express as px
import seaborn as sns
import torch
import torch.nn.functional as F
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import procrustes
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.manifold import TSNE, LocallyLinearEmbedding
from sklearn.random_projection import GaussianRandomProjection
from torch import nn
from torch.utils.data import Subset

from nnscaling import aesthetics
from nnscaling.data import (
    DatasetConfig,
    create_data_loader,
    get_dataset_class,
)
from nnscaling.models import MLP, Autoencoder, ScaledModel
from nnscaling.utils import (
    compute_model_accuracy,
    get_device,
    safetensors_metadata_parser,
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
model_name = "mnist_model"
scale_idx = "2"

In [35]:
model_path = f"/home/nsa325/work/nnscaling/model_saves/{model_name}.safetensors"
scaled_file_path = f"/home/nsa325/work/nnscaling/model_saves/scaling/loc_{scale_idx}_scaled_{model_name}.safetensors"
ae_file_path = f"/home/nsa325/work/nnscaling/model_saves/scaling/loc_{scale_idx}_autoencoder_{model_name}.safetensors"

In [36]:
# Dataset config
metadata = safetensors_metadata_parser(file_path=model_path)
dataset_config = DatasetConfig(
    dataset_name=metadata["dataset"],
    num_samples=3_000,
    split="train",
    seed=42,
)
DatasetClass = get_dataset_class(name=dataset_config.dataset_name)
train_dataset = DatasetClass(config=dataset_config)

In [37]:
# Load models
model = MLP.load_model(model_path, in_features=train_dataset.in_features).eval()
original_scaled_model = ScaledModel.load_model(model, scaled_file_path).eval()
autoencoder = Autoencoder.load_model(ae_file_path)

In [38]:
target_class = 1
idx = torch.where(train_dataset.labels == target_class)[0]
subset = Subset(train_dataset, idx)
loader = torch.utils.data.DataLoader(train_dataset, batch_size=10_000, shuffle=True)
data, labels = next(iter(loader))

In [39]:
# Compute initial PCA
def compute_reference_bases(data):
    # Compute PCA reference basis
    pca = PCA(n_components=3)
    ref = pca.fit_transform(data)
    return ref


# Function to align using Procrustes from scipy
def align_using_procrustes(reference_points, new_points):
    _, new_points_aligned, _ = procrustes(reference_points, new_points)
    return new_points_aligned


def pad_act(x, autoencoder):
    target_size = autoencoder.encoder[0].in_features
    current_size = x.size(1)
    if current_size < target_size:
        pad_size = target_size - current_size
        x = F.pad(x, (0, pad_size), mode="constant", value=0)

    return x


def replace_relu(module, slope, layer_index):
    """
    Recursively replaces ReLU activation with LeakyReLU in a model based on the given index.

    Args:
        module (nn.Module): The model to traverse.
        slope (float): The negative slope for LeakyReLU.
        layer_index (int): The index of the layer to replace.
    """
    current_index = 0

    def traverse_and_replace(submodule, counter):
        for name, child in submodule.named_children():
            if isinstance(child, nn.Sequential) or isinstance(child, nn.Module):
                counter = traverse_and_replace(child, counter)
            if isinstance(child, nn.ReLU):
                if counter == layer_index:
                    setattr(submodule, name, nn.LeakyReLU(slope))
                counter += 1
        return counter

    traverse_and_replace(module, current_index)

In [ ]:
# Enable Panel for Jupyter
pn.extension()


def create_3d_scatter_plot(data, labels, axis_range):
    x, y, z = data[:, 0], data[:, 1], data[:, 2]

    str_labels = [str(label) for label in labels]
    pca_scalar_field = np.linalg.norm(ref_a, axis=1)

    color = str_labels
    # color = pca_scalar_field
    # color_continuous_scale="Viridis")
    fig = px.scatter_3d(x=x, y=y, z=z, color=color)

    fig.update_traces(marker=dict(size=1))
    fig.update_layout(
        scene=dict(
            xaxis=dict(range=axis_range),
            yaxis=dict(range=axis_range),
            zaxis=dict(range=axis_range),
            aspectmode="cube",
            aspectratio=dict(x=1, y=1, z=1),
        ),
        showlegend=False,
    )
    return fig


def process_pca_and_align(data, reference):
    """Applies PCA, aligns using Procrustes, and returns aligned data."""
    pca = PCA(n_components=3)
    pca_result = pca.fit_transform(data)
    aligned_result = align_using_procrustes(reference, pca_result)
    return aligned_result


def update_plots(data_a, data_b, data_c, ref_a, ref_b, ref_c, labels):
    """Updates PCA and RP plots with the given data and references."""
    pca_axis_range = [-0.05, 0.05]  # Default axis range

    # First plot: PCA
    aligned_pca_result = process_pca_and_align(data_a, ref_a)
    first_fig = create_3d_scatter_plot(aligned_pca_result, labels, pca_axis_range)

    # Second plot: PCA
    aligned_pca_result = process_pca_and_align(data_b, ref_b)
    second_fig = create_3d_scatter_plot(aligned_pca_result, labels, pca_axis_range)

    # Second plot: PCA
    aligned_pca_result = process_pca_and_align(data_c, ref_c)
    third_fig = create_3d_scatter_plot(aligned_pca_result, labels, pca_axis_range)

    return second_fig, third_fig


scaled_model = deepcopy(original_scaled_model)
scaled_model.hook_model(True, True, False)
_ = scaled_model.forward_scaled(data)

scaled_act_dict = scaled_model.get_activations(detach=True)
new_keys = range(len(scaled_act_dict))
scaled_act_dict = OrderedDict(zip(new_keys, scaled_act_dict.values()))
ref_a = compute_reference_bases(scaled_act_dict[0])

decoded_act = (
    autoencoder(x=scaled_act_dict[0], k=scaled_model.num_scaled).predictions.detach().numpy()
)
decoded_act_dict = OrderedDict(zip(new_keys, decoded_act))
ref_decoded = compute_reference_bases(decoded_act_dict[0])


embedded_act = [autoencoder.encoder(scaled_act_dict[0]).detach()] * (scaled_model.num_scaled + 1)
embedded_act = [
    act
    if i == 0
    else reduce(lambda x, _: x @ autoencoder.koopman_matrix.linear.weight, range(i), act)
    for i, act in enumerate(embedded_act)
]
embedded_act = [act.detach().numpy() for act in embedded_act]
embedded_act_dict = OrderedDict(zip(new_keys, embedded_act))
ref_embedded = compute_reference_bases(embedded_act[0])

# Create sliders
slider = pn.widgets.FloatSlider(name="LeakyReLU", start=-0.5, end=0.5, step=0.01, value=0.0)
layer_select = pn.widgets.IntSlider(
    name="Layer Selector", start=new_keys.start, end=new_keys.stop - 1, step=1, value=0
)


@pn.depends(slider.param.value, layer_select.param.value)
def view(relu_param, layer_index):
    scaled_model = deepcopy(original_scaled_model)
    replace_relu(scaled_model, relu_param, layer_index)
    scaled_model.hook_model(True, True, True)
    _ = scaled_model.forward_scaled(data.flatten(start_dim=1))

    scaled_act_dict = scaled_model.get_activations(detach=True)
    new_keys = range(len(scaled_act_dict))
    scaled_act_dict = OrderedDict(zip(new_keys, scaled_act_dict.values()))

    figs = update_plots(
        scaled_act_dict[layer_index],
        decoded_act_dict[layer_index],
        embedded_act_dict[layer_index],
        ref_a,
        ref_decoded,
        ref_embedded,
        labels,
    )
    panes = [pn.pane.Plotly(fig) for fig in figs]

    return pn.Row(*panes, align="center")


# Layout
layout = pn.Column(
    pn.Row(slider, align="center"),
    pn.Row(layer_select, align="center"),
    view,
    align="center",
    sizing_mode="stretch_width",
)

layout.show()

Launching server at http://localhost:35039
